# Garmin FIT Payload Exploration

Purpose:

- Import raw Garmin running activity files in FIT format.
- Parse FIT messages into three analytical entities:
  - `sessions`: one row per run, used for run-level metrics.
  - `records`: second-by-second telemetry, used for pace, heart rate, cadence, elevation, and split-style analysis.
  - `events`: activity event messages, used to identify timer boundaries and recovery heart rate.
- Inspect available fields across sessions, records, and events.
- Validate which fields are reliable enough for bronze ingestion.
- Save representative exploration outputs for schema design.

This notebook is exploratory. Reusable Garmin authentication, download, parsing, and file IO logic should live in `ingest/garmin/**`; the notebook should only orchestrate discovery and inspect outputs.

In [1]:
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)

In [2]:
from ingest.garmin.paths import get_garmin_fit_dir
from ingest.garmin.parser import parse_fit_files

fit_dir = get_garmin_fit_dir()
fit_paths = sorted(fit_dir.glob("*.fit"))

print(f"FIT directory: {fit_dir}")
print(f"FIT files found: {len(fit_paths)}")

if len(fit_paths) == 0:
    print("No FIT files found. Exiting.")
    print("Run:")
    print("    uv run python scripts/download_garmin_fit.py")
else:
    print("Parsing FIT files...")
    parsed = parse_fit_files(fit_paths)
    sessions = parsed["sessions"]
    events = parsed["events"]
    records = parsed["records"]
    print(f"Parsed {len(records)} records from {len(fit_paths)} FIT files.")

FIT directory: /Users/frank/running-signals/data/raw/garmin/fit
FIT files found: 73
Parsing FIT files...
Parsed 273930 records from 73 FIT files.


In [3]:
entity_shapes = pd.DataFrame(
    [
        {
            "entity": "sessions",
            "rows": len(sessions),
            "columns": len(sessions.columns),
            "unique_runs": sessions["run_id"].nunique() if "run_id" in sessions.columns else None,
        },
        {
            "entity": "events",
            "rows": len(events),
            "columns": len(events.columns),
            "unique_runs": events["run_id"].nunique() if "run_id" in events.columns else None,
        },
        {
            "entity": "records",
            "rows": len(records),
            "columns": len(records.columns),
            "unique_runs": records["run_id"].nunique() if "run_id" in records.columns else None,
        },
    ]
)

entity_shapes

,entity,rows,columns,unique_runs
0,sessions,73,39,73
1,events,759,5,73
2,records,273930,19,73


### Validate field reliability

In [4]:
def column_inventory(df: pd.DataFrame, entity: str) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "entity": entity,
            "column": df.columns,
            "dtype": [str(dtype) for dtype in df.dtypes],
            "non_null_count": df.notna().sum().values,
            "null_count": df.isna().sum().values,
            "null_pct": [df[column].isna().mean() * 100 for column in df.columns],
            "unique_count": [df[column].nunique(dropna=True) for column in df.columns],
        }
    ).sort_values(["null_pct", "column"])

field_inventory = pd.concat(
    [
        column_inventory(sessions, "sessions"),
        column_inventory(events, "events"),
        column_inventory(records, "records"),
    ],
    ignore_index=True,
)

# field_inventory

In [5]:
candidate_fields = {
    "sessions": [
        "run_id",
        "timestamp",
        "start_time",
        "total_elapsed_time",
        "total_timer_time",
        "total_distance",
        "sport",
        "sub_sport",
        "avg_heart_rate",
        "max_heart_rate",
        "enhanced_avg_speed",
        "avg_speed",
        "enhanced_max_speed",
        "max_speed",
        "total_ascent",
        "total_descent",
        "total_calories",
        "total_training_effect",
        "total_anaerobic_training_effect",
        "avg_cadence",
        "max_cadence",
        "avg_running_cadence",
        "max_running_cadence",
    ],
    "events": [
        "run_id",
        "timestamp",
        "event",
        "event_type",
        "data",
    ],
    "records": [
        "run_id",
        "timestamp",
        "distance",
        "heart_rate",
        "enhanced_speed",
        "enhanced_altitude",
        "cadence",
        "temperature",
        "position_lat",
        "position_long",
        "position_lat_deg",
        "position_long_deg",
        "stance_time",
        "vertical_oscillation",
        "vertical_ratio",
        "step_length",
    ],
}

In [6]:
candidate_inventory = pd.concat(
    [
        field_inventory[
            field_inventory["entity"].eq(entity)
            & field_inventory["column"].isin(fields)
        ]
        for entity, fields in candidate_fields.items()
    ],
    ignore_index=True,
)

In [7]:
FIELD_CLASSIFICATION_RULES = {
    "required_max_null_pct": 0.0,
    "recommended_max_null_pct": 20.0,
    "optional_max_null_pct": 80.0,
}

def classify_field(null_pct: float) -> str:
    if null_pct == 0:
        return "required_or_stable"
    if null_pct <= FIELD_CLASSIFICATION_RULES["recommended_max_null_pct"]:
        return "recommended"
    if null_pct <= FIELD_CLASSIFICATION_RULES["optional_max_null_pct"]:
        return "optional"
    return "exclude_now"

candidate_inventory = candidate_inventory.copy()
candidate_inventory["reliability_class"] = candidate_inventory["null_pct"].apply(classify_field)

candidate_inventory = candidate_inventory.sort_values(["entity", "reliability_class", "null_pct", "column"]).reset_index(drop=True)
candidate_inventory

,entity,column,dtype,non_null_count,null_count,null_pct,unique_count,reliability_class
0,events,data,int64,759,0,0.000000,32,required_or_stable
1,events,event,object,759,0,0.000000,2,required_or_stable
2,events,event_type,object,759,0,0.000000,3,required_or_stable
3,events,run_id,object,759,0,0.000000,73,required_or_stable
4,events,timestamp,"datetime64[ns, UTC]",759,0,0.000000,759,required_or_stable
5,records,temperature,float64,218999,54931,20.052933,24,optional
6,records,enhanced_speed,float64,273926,4,0.001460,447,recommended
7,records,position_lat,float64,273760,170,0.062060,216243,recommended
8,records,position_lat_deg,float64,273760,170,0.062060,216243,recommended
9,records,position_long,float64,273760,170,0.062060,209955,recommended


### Validate sessions

In [8]:
session_quality_checks = {
    "session_rows": len(sessions),
    "unique_runs": sessions["run_id"].nunique(),
    "duplicate_run_ids": int(sessions["run_id"].duplicated().sum()),
}

session_quality_checks

{'session_rows': 73, 'unique_runs': 73, 'duplicate_run_ids': 0}

In [9]:
session_preview_columns = [
    column
    for column in [
        "run_id",
        "start_time",
        "timestamp",
        "sport",
        "sub_sport",
        "total_distance",
        "total_timer_time",
        "total_elapsed_time",
        "avg_heart_rate",
        "max_heart_rate",
        "enhanced_avg_speed",
        "avg_speed",
        "total_ascent",
        "total_descent",
    ]
    if column in sessions.columns
]

sessions[session_preview_columns].head()

,run_id,start_time,timestamp,sport,sub_sport,total_distance,total_timer_time,total_elapsed_time,avg_heart_rate,max_heart_rate,enhanced_avg_speed,total_ascent,total_descent
0,21523624126,2026-01-12 15:22:46+00:00,2026-01-12 15:22:46+00:00,running,generic,6440.24,2106.654,2441.664,160,184,3.057,35,37
1,21567467974,2026-01-16 16:20:00+00:00,2026-01-16 16:20:00+00:00,running,generic,5255.77,1803.614,1811.746,153,170,2.914,28,28
2,21609305672,2026-01-20 15:37:10+00:00,2026-01-20 15:37:10+00:00,running,generic,5006.70,1576.563,1598.712,168,188,3.176,42,37
3,21641401109,2026-01-23 15:46:01+00:00,2026-01-23 15:46:01+00:00,running,generic,10059.59,3300.366,3300.366,164,182,3.048,99,110
4,21697851224,2026-01-29 06:10:37+00:00,2026-01-29 06:10:37+00:00,running,generic,5009.42,1561.594,1594.747,169,181,3.208,37,35


### Validate records

In [10]:
records_per_run = (
    records.groupby("run_id")
    .size()
    .reset_index(name="record_count")
    .sort_values("record_count", ascending=False)
)

records_per_run.describe()

,record_count
count,73.000000
mean,3752.465753
std,1585.466529
min,401.000000
25%,2737.000000
50%,3593.000000
75%,4712.000000
max,8546.000000


In [11]:
record_quality_checks = {
    "record_rows": len(records),
    "unique_runs": records["run_id"].nunique(),
    "timestamp_null_pct": f"{records['timestamp'].isna().mean() * 100:.4f}%" if "timestamp" in records.columns else None,
    "distance_null_pct": f"{records['distance'].isna().mean() * 100:.4f}%" if "distance" in records.columns else None,
    "heart_rate_null_pct": f"{records['heart_rate'].isna().mean() * 100:.4f}%" if "heart_rate" in records.columns else None,
    "enhanced_speed_null_pct": f"{records['enhanced_speed'].isna().mean() * 100:.4f}%" if "enhanced_speed" in records.columns else None,
}

record_quality_checks

{'record_rows': 273930,
 'unique_runs': 73,
 'timestamp_null_pct': '0.0000%',
 'distance_null_pct': '0.0000%',
 'heart_rate_null_pct': '0.0000%',
 'enhanced_speed_null_pct': '0.0015%'}

### Validate events and recovery HR

In [12]:
event_counts = (
    events.groupby(["event", "event_type"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

event_counts

,event,event_type,count
1,timer,start,349
2,timer,stop_all,345
0,recovery_hr,marker,65


In [13]:
recovery_events = events[
    events["event"].astype(str).str.contains("recovery_hr", case=False, na=False)
].copy().reset_index(drop=True)

if "data" in recovery_events.columns:
    recovery_events["recovery_hr"] = pd.to_numeric(
        recovery_events["data"],
        errors="coerce",
    )

recovery_events.head()

,timestamp,data,event,event_type,run_id,recovery_hr
0,2026-01-31 05:18:49+00:00,137,recovery_hr,marker,21716304908,137
1,2026-02-03 09:32:20+00:00,128,recovery_hr,marker,21748778514,128
2,2026-02-14 15:54:46+00:00,113,recovery_hr,marker,21867138627,113
3,2026-02-16 10:53:24+00:00,99,recovery_hr,marker,21883871011,99
4,2026-02-17 10:14:24+00:00,107,recovery_hr,marker,21894036001,107


In [14]:
recovery_event_count_distribution = (
    recovery_events.groupby("run_id")
    .size()
    .value_counts()
    .rename_axis("recovery_event_count")
    .reset_index(name="run_count")
    .sort_values("recovery_event_count")
    .reset_index(drop=True)
)

recovery_event_count_distribution

,recovery_event_count,run_count
0,1,48
1,2,7
2,3,1


In [15]:
recovery_coverage = {
    "runs_with_recovery_events": recovery_events["run_id"].nunique(),
    "recovery_coverage_pct": f"{(recovery_events['run_id'].nunique() / sessions['run_id'].nunique() * 100):.2f}%"
    if sessions["run_id"].nunique() > 0
    else None,
}

recovery_coverage

{'runs_with_recovery_events': 56, 'recovery_coverage_pct': '76.71%'}

In [16]:
last_record_per_run = (
    records.sort_values(["run_id", "timestamp"])
    .groupby("run_id", as_index=False)
    .tail(1)
)

last_record_columns = [
    column
    for column in ["run_id", "timestamp", "heart_rate", "distance"]
    if column in last_record_per_run.columns
]

last_records = last_record_per_run[last_record_columns].rename(
    columns={
        "timestamp": "last_record_timestamp",
        "heart_rate": "final_record_heart_rate",
        "distance": "final_record_distance",
    }
)

last_records.head()

,run_id,last_record_timestamp,final_record_heart_rate,final_record_distance
2120,21523624126,2026-01-12 16:03:28+00:00,172,6440.24
3926,21567467974,2026-01-16 16:50:12+00:00,159,5255.77
5506,21609305672,2026-01-20 16:03:49+00:00,180,5006.70
8807,21641401109,2026-01-23 16:41:01+00:00,171,10059.59
10371,21697851224,2026-01-29 06:37:12+00:00,173,5009.42


In [17]:
if recovery_events.empty:
    print("No recovery events found. Cannot validate recovery HR drop.")
elif "recovery_hr" not in recovery_events.columns:
    print("recovery_events exists, but `recovery_hr` has not been derived yet.")
    display(recovery_events.head(20))
else:
    recovery_validation = recovery_events.merge(last_records, on="run_id", how="left")

    if "final_record_heart_rate" in recovery_validation.columns:
        recovery_validation["final_hr_minus_recovery_hr"] = (
            recovery_validation["final_record_heart_rate"]
            - recovery_validation["recovery_hr"]
        )

    display(
        recovery_validation[
            [
                column
                for column in [
                    "run_id",
                    "last_record_timestamp",
                    "recovery_hr",
                    "final_record_heart_rate",
                    "final_hr_minus_recovery_hr",
                    "final_record_distance",
                ]
                if column in recovery_validation.columns
            ]
        ].head()
    )

,run_id,last_record_timestamp,recovery_hr,final_record_heart_rate,final_hr_minus_recovery_hr,final_record_distance
0,21716304908,2026-01-31 05:16:49+00:00,137,178,41,11367.67
1,21748778514,2026-02-03 09:30:20+00:00,128,175,47,5296.06
2,21867138627,2026-02-14 15:52:46+00:00,113,142,29,16010.17
3,21883871011,2026-02-16 10:51:24+00:00,99,149,50,9358.23
4,21894036001,2026-02-17 10:12:24+00:00,107,144,37,7173.27


### Save representative exploration outputs

In [18]:
from ingest.garmin.paths import get_garmin_exploration_dir

exploration_dir = get_garmin_exploration_dir()
print(exploration_dir)

/Users/frank/running-signals/data/raw/garmin/exploration


#### field inventory

In [19]:
field_inventory_path = exploration_dir / "fit_field_inventory.csv"
candidate_inventory_path = exploration_dir / "fit_candidate_field_inventory.csv"

field_inventory.to_csv(field_inventory_path, index=False)
candidate_inventory.to_csv(candidate_inventory_path, index=False)

print(f"Field inventory saved to: {field_inventory_path}")
print(f"Candidate field inventory saved to: {candidate_inventory_path}")

Field inventory saved to: /Users/frank/running-signals/data/raw/garmin/exploration/fit_field_inventory.csv
Candidate field inventory saved to: /Users/frank/running-signals/data/raw/garmin/exploration/fit_candidate_field_inventory.csv


#### representative samples

In [20]:
sessions_sample_path = exploration_dir / "sessions.sample.csv"
events_sample_path = exploration_dir / "events.sample.csv"
records_sample_path = exploration_dir / "records.sample.csv"

sessions.head(100).to_csv(sessions_sample_path, index=False)
events.head(500).to_csv(events_sample_path, index=False)
records.head(5000).to_csv(records_sample_path, index=False)

print(f"Sessions sample dataset saved to: {sessions_sample_path}")
print(f"Events sample dataset saved to: {events_sample_path}")
print(f"Records sample dataset saved to: {records_sample_path}")

Sessions sample dataset saved to: /Users/frank/running-signals/data/raw/garmin/exploration/sessions.sample.csv
Events sample dataset saved to: /Users/frank/running-signals/data/raw/garmin/exploration/events.sample.csv
Records sample dataset saved to: /Users/frank/running-signals/data/raw/garmin/exploration/records.sample.csv


#### recovery HR sample

In [21]:
recovery_events_path = exploration_dir / "recovery_events.sample.csv"

recovery_events.to_csv(recovery_events_path, index=False)

print(f"Recovery events dataset saved to: {recovery_events_path}")

Recovery events dataset saved to: /Users/frank/running-signals/data/raw/garmin/exploration/recovery_events.sample.csv


#### schema recommendation draft

In [22]:
schema_recommendation = candidate_inventory[
    [
        "entity",
        "column",
        "dtype",
        "non_null_count",
        "null_pct",
        "unique_count",
        "reliability_class",
    ]
].sort_values(["entity", "reliability_class", "null_pct", "column"])

schema_recommendation_path = exploration_dir / "fit_bronze_schema_recommendation.csv"

schema_recommendation.to_csv(schema_recommendation_path, index=False)

print(f"Schema recommendation saved to: {schema_recommendation_path}")

Schema recommendation saved to: /Users/frank/running-signals/data/raw/garmin/exploration/fit_bronze_schema_recommendation.csv


#### run coverage summary

In [23]:
coverage_summary = pd.DataFrame(
    [
        {
            "metric": "session_rows",
            "value": len(sessions),
        },
        {
            "metric": "session_unique_runs",
            "value": sessions["run_id"].nunique(),
        },
        {
            "metric": "event_rows",
            "value": len(events),
        },
        {
            "metric": "event_unique_runs",
            "value": events["run_id"].nunique(),
        },
        {
            "metric": "record_rows",
            "value": len(records),
        },
        {
            "metric": "record_unique_runs",
            "value": records["run_id"].nunique(),
        },
        {
            "metric": "recovery_event_rows",
            "value": len(recovery_events),
        },
        {
            "metric": "recovery_event_unique_runs",
            "value": recovery_events["run_id"].nunique(),
        },
    ]
)

coverage_summary_path = exploration_dir / "fit_coverage_summary.csv"

coverage_summary.to_csv(coverage_summary_path, index=False)

print(f"Coverage summary saved to: {coverage_summary_path}")

Coverage summary saved to: /Users/frank/running-signals/data/raw/garmin/exploration/fit_coverage_summary.csv


## Exploration conclusion

This notebook confirms that Garmin FIT files can be parsed into three useful analytical entities:

- `sessions`: run-level facts suitable for consistency, volume, and summary fitness metrics.
- `records`: high-frequency telemetry suitable for pace/heart-rate analysis, splits, and within-run patterns.
- `events`: event messages suitable for timer state changes and recovery heart rate extraction.

The next engineering step is to define bronze tables for:

- `bronze_garmin_fit_sessions`
- `bronze_garmin_fit_records`
- `bronze_garmin_fit_events`

Bronze ingestion should preserve source-level values with minimal transformation. Metric derivation such as pace, HR-band efficiency, and recovery HR drop should happen in silver or gold models.